# ESMFold structure-validation of ProteinMPNN-designed sequences

Optional Phase 3 extension ("only if compute and time allow" per the project roadmap). This notebook folds a handful of sequences designed by ProteinMPNN (full-backbone model, `v_48_020`, T=0.1, seed=37, sample 1) and compares the predicted structure's confidence (pLDDT) and, where feasible, backbone RMSD against the native structure.

**How to run on Kaggle:**
1. Create a new Kaggle Notebook, upload this file (File > Import Notebook).
2. Settings > Accelerator > GPU (T4 x2 or P100).
3. Run all cells. First run downloads the ESMFold weights (~2.7GB), takes a few minutes.

Results (pLDDT, CA RMSD) are in `results/` alongside this notebook.

In [ ]:
# --no-deps: Kaggle's image ships a torch build already matched to the
# assigned GPU's CUDA architecture. A dependency-resolved install of
# transformers/accelerate can otherwise pull in a replacement torch wheel
# that lacks compiled kernels for that GPU ('no kernel image available').
!pip install -q --no-deps fair-esm transformers accelerate
!pip install -q biopython
import torch
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# 5 short designed sequences (full-backbone model, T=0.1, sample 1) and their
# native counterparts, pulled from reproduction/phase2_reproduction/fullbackbone/seqs/
# and reproduction/phase1_data_prep/native_sequences.fasta

sequences = {
    '1UBQ': {
        'designed': 'MTIFVAREDGTTLELEVEPSDTIAELKKKIEEKTGIPPEEQKLIYKGKVLEDEKTLADYNIEEGDTIKLELVPKGG',
        'native':   'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG',
        'seq_recovery_reported': 0.5526,
    },
    '1VII': {
        'designed': 'KPTPEEAKKLLGGTVEEFKKLSPEEQEEAYKKNLAK',
        'native':   'MLSDEDFKAVFGMTRSAFANLPLWKQQNLKKEKGLF',
        'seq_recovery_reported': 0.2222,
    },
    '2GB1': {
        'designed': 'KKYKVEIEGKNYTGSFTVEAKNKEEAKEKVKEELKKYGVEGEFYFDEENNTFKVKD',
        'native':   'MTYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE',
        'seq_recovery_reported': 0.3214,
    },
    '1CRN': {
        'designed': 'TVCCPSKEARDKYLECLKPGTPKEECAKATGCIIIPGTTCPADYPY',
        'native':   'TTCCPSIVARSNFNVCRLPGTPEAICATYTGCIIIPGATCPGDYAN',
        'seq_recovery_reported': 0.5870,
    },
    '1ENH': {
        'designed': 'APPLTFSAEQRAALDARFARNPELSDEELAALSAELGLPAEQIRAWFAARRAAA',
        'native':   'RPRTAFSSEQLARLKREFNENRYLTERRRQQLSSELGLNEAQIKIWFQNKRAKI',
        'seq_recovery_reported': 0.4074,
    },
}
for k, v in sequences.items():
    assert len(v['designed']) == len(v['native'])
print('Loaded', len(sequences), 'sequence pairs')

In [ ]:
from transformers import AutoTokenizer, EsmForProteinFolding
import torch

tokenizer = AutoTokenizer.from_pretrained('facebook/esmfold_v1')
model = EsmForProteinFolding.from_pretrained('facebook/esmfold_v1', low_cpu_mem_usage=True)

# Some Kaggle GPU assignments (observed: Tesla P100, compute capability 6.0)
# are older than the minimum architecture (7.0+) the pre-installed torch build
# was compiled for. A matmul probe was NOT sufficient to detect this (cuBLAS
# matmul works via driver JIT, but transformers/esmfold's element-wise and
# custom kernels are hard-compiled for sm_70+ only and fail regardless) --
# check the compute capability directly instead of probing with an operation.
use_cuda = False
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}  compute capability: {major}.{minor}')
    if major >= 7:
        use_cuda = True
    else:
        print(f'Compute capability {major}.{minor} < 7.0 required by this torch build -- using CPU instead')
print('Using device:', 'cuda' if use_cuda else 'cpu')
model = model.cuda() if use_cuda else model
# fp16 skipped: caused 'no kernel image available' on some Kaggle GPU images; fp32 is fine for these tiny (<80 residue) sequences
model.trunk.set_chunk_size(64)
print('Model loaded')

In [ ]:
def fold(seq):
    tok = tokenizer([seq], return_tensors='pt', add_special_tokens=False)
    if use_cuda:
        tok = {k: v.cuda() for k, v in tok.items()}
    with torch.no_grad():
        out = model(tok['input_ids'])
    mean_plddt = out['plddt'][0, :, 1].mean().item() * 100  # CA pLDDT, 0-100 scale
    pdb_str = model.output_to_pdb(out)[0]
    return mean_plddt, pdb_str

results = {}
for pdb_id, v in sequences.items():
    print(f'Folding {pdb_id} designed...')
    plddt_designed, pdb_designed = fold(v['designed'])
    print(f'Folding {pdb_id} native...')
    plddt_native, pdb_native = fold(v['native'])
    results[pdb_id] = {
        'plddt_designed': plddt_designed,
        'plddt_native': plddt_native,
        'seq_recovery_reported': v['seq_recovery_reported'],
    }
    with open(f'{pdb_id}_designed_esmfold.pdb', 'w') as f:
        f.write(pdb_designed)
    with open(f'{pdb_id}_native_esmfold.pdb', 'w') as f:
        f.write(pdb_native)
    print(f'{pdb_id}: designed pLDDT={plddt_designed:.1f}  native pLDDT={plddt_native:.1f}')

In [ ]:
# Backbone RMSD between ESMFold(designed) and ESMFold(native) CA traces --
# a proxy for "did the designed sequence fold into (approximately) the same
# shape as the native protein", which is the actual claim ProteinMPNN makes
# (not that the sequence matches, but that it folds correctly).
from Bio.PDB import PDBParser
from Bio.PDB.qcprot import QCPSuperimposer
import numpy as np

parser = PDBParser(QUIET=True)

def get_ca_coords(path):
    s = parser.get_structure('x', path)
    return np.array([res['CA'].coord for res in s[0]['A'] if 'CA' in res])

print(f"{'pdb_id':<8}{'plddt_designed':<16}{'plddt_native':<15}{'ca_rmsd_A':<12}{'seq_recovery'}")
for pdb_id in sequences:
    ca_d = get_ca_coords(f'{pdb_id}_designed_esmfold.pdb')
    ca_n = get_ca_coords(f'{pdb_id}_native_esmfold.pdb')
    n = min(len(ca_n), len(ca_d))
    sup = QCPSuperimposer()
    sup.set(ca_n[:n], ca_d[:n])
    sup.run()
    rmsd = sup.get_rms()
    r = results[pdb_id]
    print(f"{pdb_id:<8}{r['plddt_designed']:<16.1f}{r['plddt_native']:<15.1f}{rmsd:<12.2f}{r['seq_recovery_reported']:.4f}")

## Interpreting the output

- **High pLDDT for the designed sequence** (roughly comparable to the native's pLDDT) suggests ESMFold is confident the designed sequence folds into a well-defined structure — consistent with the paper's core claim, independent of raw sequence identity.
- **Low CA RMSD to the native fold** (a few Å for small domains) would support that the designed sequence actually reproduces the *target backbone*, which is the real design objective (sequence recovery is a proxy metric, not the goal itself).
- Note this is a **secondary, independent** structure predictor (ESMFold) checking ProteinMPNN's designs — not a wet-lab validation, and pLDDT/RMSD from a single predicted structure per sequence is a coarse signal, not a rigorous validation. Treat as a sanity check, not proof.